In [0]:
import sys
import pandas as pd 
from datetime import datetime, timedelta
from pyspark.sql.functions import min as spark_min, max as spark_max

# Optimize shuffle for your 2-core single node
spark.conf.set("spark.sql.shuffle.partitions", 2)

# 1. Widget Setup
dbutils.widgets.text("customer_id", "cust_001")
dbutils.widgets.text("object_name", "events")
dbutils.widgets.text("source_system", "bigquery")
dbutils.widgets.text("bucket_path", "s3://hgi-databricks-data-lakehouse-dev/")
dbutils.widgets.text("bq_native_table", "v4c-bigquery.v4c_bigquery_dataset.event_raw")

customer_id = dbutils.widgets.get("customer_id")
object_name = dbutils.widgets.get("object_name")
source_system = dbutils.widgets.get("source_system")
bucket_path = dbutils.widgets.get("bucket_path")
bq_native_table = dbutils.widgets.get("bq_native_table")

now = datetime.now()
yyyy, mm, dd, hh = now.strftime("%Y"), now.strftime("%m"), now.strftime("%d"), now.strftime("%H")
historical_s3_path = f"{bucket_path}landing-zone/{source_system}/{customer_id}/{object_name}/historical/{yyyy}/{mm}/{dd}/{hh}"

print(f"🚀 Starting FAST NATIVE BigQuery Historical Load for {customer_id} - {object_name}")

gcp_project_id = bq_native_table.split(".")[0]

# 2. Securely Retrieve GCP Credentials
try:
    gcp_secret = dbutils.secrets.get(scope="gcp_auth", key="bq_key")
except Exception as e:
    raise Exception("❌ Failed to retrieve GCP credentials. Ensure the secret scope 'gcp_auth' and key 'bq_key' exist.")

try:
    # 3. Find Min/Max (Direct gRPC Read)
    print("Fetching timeline boundaries from BigQuery...")
    bounds_df = spark.read \
        .format("bigquery") \
        .option("credentials", gcp_secret) \
        .option("parentProject", gcp_project_id) \
        .option("project", gcp_project_id) \
        .load(bq_native_table) \
        .select(
            spark_min("event_timestamp").alias("min_ts"),
            spark_max("event_timestamp").alias("max_ts")
        )
        
    bounds = bounds_df.collect()[0]
    start_date_raw = bounds["min_ts"]
    end_date_raw = bounds["max_ts"]
    
    if start_date_raw is None or end_date_raw is None:
        print("No data found in source.")
        dbutils.notebook.exit("0") 
    
    start_date = pd.to_datetime(start_date_raw).replace(tzinfo=None)
    end_date = pd.to_datetime(end_date_raw).replace(tzinfo=None)
        
    print(f"Data range: {start_date} to {end_date}")

    # -------------------------------------------------------------------------
    # THE FIX: Increase interval to 12 hours to drastically reduce BQ scans
    # -------------------------------------------------------------------------
    intervals = []
    current_start = start_date
    chunk_interval = timedelta(hours=12) 
    
    while current_start <= (end_date + chunk_interval):
        current_end = current_start + chunk_interval
        
        # Formatting to exact ISO string for direct String-to-String comparison
        str_start = current_start.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"
        str_end = current_end.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"
        
        intervals.append((str_start, str_end))
        current_start = current_end

    print(f"Divided timeline into {len(intervals)} 12-hour chunks. Processing sequentially...")

    # 5. Process Sequentially using Native Connector
    for start_str, end_str in intervals:
        try:
            print(f"Processing chunk: {start_str} to {end_str}...")
            
            # String comparison filter because BQ column is now a STRING
            bq_filter_logic = f"event_timestamp >= '{start_str}' AND event_timestamp < '{end_str}'"
            
            df_chunk = spark.read \
                .format("bigquery") \
                .option("credentials", gcp_secret) \
                .option("parentProject", gcp_project_id) \
                .option("project", gcp_project_id) \
                .option("filter", bq_filter_logic) \
                .load(bq_native_table)
            
            # Repartitioning to 2 perfectly aligns with your 2-core m5d.large instance
            df_chunk.repartition(2).write.mode("append").format("parquet").option("compression","snappy").save(historical_s3_path)
            print(f"✅ Successfully written chunk: {start_str}")
            
        except Exception as e:
            print(f"❌ Failed chunk {start_str}: {str(e)}")
            raise e

    print("✅ Historical Data Load FINISHED!")

    # 6. Initialize Watermark with 1-Minute Lookback
    print(f"Initializing watermark to {end_date} minus 1 minute...")
    
    spark.sql("""
        CREATE TABLE IF NOT EXISTS ingestion_metadata.watermarks (
            source_system STRING,
            object_name STRING,
            last_processed_timestamp TIMESTAMP
        ) USING DELTA
    """)
    
    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW new_watermarks AS
        SELECT source_system, object_name, last_processed_timestamp
        FROM ingestion_metadata.watermarks
        WHERE NOT (source_system = '{source_system}' AND object_name = '{object_name}')
        UNION ALL
        SELECT '{source_system}' AS source_system, '{object_name}' AS object_name, TIMESTAMP('{end_date}') - INTERVAL 1 MINUTE AS last_processed_timestamp
    """)
    
    spark.sql("INSERT OVERWRITE TABLE ingestion_metadata.watermarks SELECT * FROM new_watermarks")
    print("✅ Watermark initialized successfully.")
    
    dbutils.notebook.exit("Success")

except Exception as e:
    print(f"❌ Pipeline Failed: {str(e)}")
    raise e